# Find Film Locations With Qdrant Score Boosting

A location scout needs **an abandoned industrial warehouse with exposed brick and tall windows**. A strong visual match may be far from the crew's base, unavailable for filming, or missing useful facilities. In this notebook, you will retrieve matches from descriptions, apply eligibility rules, and explain how facility and distance bonuses change the order.

**You will build:** semantic retrieval → permission/radius filters → a facility bonus → a geographic decay bonus → a combined ranking. You will also see a weight that overwhelms relevance, a candidate that a small prefetch misses, and a ranking that changes when the crew base moves.

**Allow 35–50 minutes**, including experiments; first-run downloads depend on your connection. Choose a standard **CPU runtime** in Google Colab. No Qdrant account, API key, GPU, paid inference service, or Docker is required. This notebook starts the actual Qdrant server as a subprocess, not Python-client local mode.

The 40 records below are **fictional teaching fixtures around Warsaw**. Their descriptions, coordinates, permissions, and facilities do not describe real venues. The data illustrates ranking mechanics; it is not a relevance benchmark or a film-location recommendation.

## 1. Install the Pinned Dependencies

The notebook uses Qdrant server **1.20.2**, Python client **1.20.0**, and FastEmbed **0.8.0**. Basic formula queries require Qdrant 1.14 or later. Run the cells in order. If you already imported different versions in this kernel, restart the session after installation.

In [1]:
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "qdrant-client==1.20.0",
        "fastembed==0.8.0",
        "pandas==2.3.3",
        "huggingface-hub==1.33.0",
        "httpx==0.28.1",
    ]
)

## 2. Start an Isolated Qdrant Server

The bootstrap downloads a pinned official release and verifies its SHA-256 before extracting only the executable. Linux x86-64 is the Google Colab path; native Apple Silicon and Intel macOS are also supported. The server listens only on localhost and stores this run's data in a temporary directory. A free port avoids interfering with another Qdrant instance.

In [2]:
import atexit
import asyncio
import hashlib
import platform
import shutil
import socket
import subprocess
import tarfile
import tempfile
from pathlib import Path

import httpx
from qdrant_client import AsyncQdrantClient, models

QDRANT_VERSION = "1.20.2"
RELEASES = {
    ("Linux", "x86_64"): (
        "qdrant-x86_64-unknown-linux-musl.tar.gz",
        "7c3e956b97830a9faa6391540330380c2fa4aa2f92df99bd8d1880a703b922af",
    ),
    ("Darwin", "arm64"): (
        "qdrant-aarch64-apple-darwin.tar.gz",
        "385b244e6283968b2be926358541b5bc19b9ada7ea9586e900684388bc912d5f",
    ),
    ("Darwin", "x86_64"): (
        "qdrant-x86_64-apple-darwin.tar.gz",
        "73a6538ccccb892ec2c6fa3431faf1e73fba7eb2179079adbd0c65ae4c55bf6a",
    ),
}
asset, expected_digest = RELEASES[(platform.system(), platform.machine())]
runtime_dir = Path(tempfile.mkdtemp(prefix="film-location-qdrant-"))
archive = runtime_dir / asset
release_url = (
    f"https://github.com/qdrant/qdrant/releases/download/v{QDRANT_VERSION}/{asset}"
)
async with httpx.AsyncClient(follow_redirects=True, timeout=120) as http:
    response = await http.get(release_url)
    response.raise_for_status()
    archive.write_bytes(response.content)
assert hashlib.sha256(archive.read_bytes()).hexdigest() == expected_digest
binary = runtime_dir / "qdrant"
with tarfile.open(archive) as bundle:
    executable = bundle.extractfile("qdrant")
    assert executable is not None
    with binary.open("wb") as output:
        shutil.copyfileobj(executable, output)
binary.chmod(0o755)

with socket.socket() as port_probe:
    port_probe.bind(("127.0.0.1", 0))
    port = port_probe.getsockname()[1]
QDRANT_URL = f"http://127.0.0.1:{port}"
config = runtime_dir / "config.yaml"
config.write_text(
    "log_level: ERROR\n"
    "storage:\n  storage_path: ./storage\n  snapshots_path: ./snapshots\n"
    f"service:\n  host: 127.0.0.1\n  http_port: {port}\n  grpc_port: null\n"
    "  enable_cors: false\ncluster:\n  enabled: false\n"
)
server_log = (runtime_dir / "server.log").open("w")
server = subprocess.Popen(
    [str(binary), "--config-path", str(config), "--disable-telemetry"],
    cwd=runtime_dir,
    stdout=server_log,
    stderr=subprocess.STDOUT,
)
atexit.register(server.terminate)
async with httpx.AsyncClient(timeout=2) as http:
    for _ in range(60):
        if server.poll() is not None:
            raise RuntimeError((runtime_dir / "server.log").read_text())
        try:
            response = await http.get(f"{QDRANT_URL}/readyz")
            if response.is_success:
                break
        except httpx.ConnectError:
            pass
        await asyncio.sleep(0.5)
    else:
        raise TimeoutError("Qdrant did not become ready; inspect server.log.")
client = AsyncQdrantClient(url=QDRANT_URL)
server_info = await client.info()
assert server_info.version == QDRANT_VERSION
print(f"Qdrant server {server_info.version} is ready at {QDRANT_URL}")

Qdrant server 1.20.2 is ready at http://127.0.0.1:39655


## 3. Load the Teaching Catalog

`description` becomes a vector. The other fields remain payload values: `permission_available` is a hard requirement, while `power_available` is a preference. Each valid `location` is a `{lat, lon}` object.

Three records deliberately expose failure modes: Old Kiln Hall is unavailable, Outer Ring Hangar is outside our radius, and Unmapped Factory has no verified coordinate. A geographic filter excludes the missing coordinate before the formula runs; we never invent a fallback city for it.

In [3]:
import json

locations = json.loads(r"""[
  {
    "id": 1,
    "name": "Old Kiln Hall",
    "description": "An abandoned industrial warehouse with weathered red brick walls, tall factory windows, steel roof trusses and an empty concrete floor. A moody space for a warehouse film scene.",
    "location": {
      "lat": 52.251,
      "lon": 21.049
    },
    "permission_available": false,
    "power_available": true
  },
  {
    "id": 2,
    "name": "East Brickworks",
    "description": "An abandoned industrial warehouse with exposed brick walls, tall dusty factory windows and rusted steel beams. Its empty concrete floor and worn surfaces suit a gritty film scene.",
    "location": {
      "lat": 52.258,
      "lon": 21.096
    },
    "permission_available": true,
    "power_available": false
  },
  {
    "id": 3,
    "name": "Riverside Workshop",
    "description": "A disused repair workshop with red brick walls, high windows, metal columns and a concrete floor. Soft daylight falls across an open industrial interior.",
    "location": {
      "lat": 52.238,
      "lon": 21.035
    },
    "permission_available": true,
    "power_available": true
  },
  {
    "id": 4,
    "name": "North Foundry",
    "description": "A deserted factory warehouse with exposed brick walls, tall windows, old steel gantries and a broad empty concrete floor. Dust and worn industrial machinery create a dramatic abandoned setting.",
    "location": {
      "lat": 52.312,
      "lon": 21.043
    },
    "permission_available": true,
    "power_available": true
  },
  {
    "id": 5,
    "name": "Glass Studio",
    "description": "A bright contemporary photography studio with white plaster walls, a seamless backdrop and a polished floor. The room has modern lighting and a clean commercial appearance.",
    "location": {
      "lat": 52.23,
      "lon": 21.013
    },
    "permission_available": true,
    "power_available": true
  },
  {
    "id": 6,
    "name": "Trackside Depot",
    "description": "A vacant railway maintenance warehouse with exposed red brick, tall arched windows and heavy metal roof supports. Faded paint and worn concrete give the hall an abandoned industrial character.",
    "location": {
      "lat": 52.278,
      "lon": 20.919
    },
    "permission_available": true,
    "power_available": false
  },
  {
    "id": 7,
    "name": "South Textile Mill",
    "description": "An empty former textile factory with exposed brick walls, rows of tall windows and old steel columns. Its expansive worn floor and weathered surfaces suit a deserted industrial film scene.",
    "location": {
      "lat": 52.126,
      "lon": 21.034
    },
    "permission_available": true,
    "power_available": true
  },
  {
    "id": 8,
    "name": "Rehearsal Loft",
    "description": "An open rehearsal loft inside a converted factory, with brick walls, large windows and timber flooring. The industrial room is tidy and still in regular use.",
    "location": {
      "lat": 52.247,
      "lon": 20.985
    },
    "permission_available": true,
    "power_available": true
  },
  {
    "id": 9,
    "name": "West Loading Hall",
    "description": "An unused distribution warehouse with grey concrete walls, low strip windows and steel loading doors. The large empty floor has tire marks and painted traffic lanes.",
    "location": {
      "lat": 52.227,
      "lon": 20.909
    },
    "permission_available": true,
    "power_available": true
  },
  {
    "id": 10,
    "name": "Brick Pump House",
    "description": "A small decommissioned pump station with red brick walls, tall narrow windows and heavy iron pipes. Its dim industrial room has patches of moss and cracked concrete.",
    "location": {
      "lat": 52.213,
      "lon": 21.05
    },
    "permission_available": true,
    "power_available": false
  },
  {
    "id": 11,
    "name": "Outer Ring Hangar",
    "description": "An abandoned industrial warehouse with exposed brick walls, tall factory windows, steel beams and an empty concrete floor. Weathered surfaces and dust create a gritty warehouse setting.",
    "location": {
      "lat": 52.409,
      "lon": 21.014
    },
    "permission_available": true,
    "power_available": true
  },
  {
    "id": 12,
    "name": "Canal Storehouse",
    "description": "A narrow old storage building with peeling brickwork, small barred windows and wooden roof beams. Crates and dusty shelves remain along the walls.",
    "location": {
      "lat": 52.282,
      "lon": 21.061
    },
    "permission_available": true,
    "power_available": false
  },
  {
    "id": 13,
    "name": "Concrete Soundstage",
    "description": "A purpose-built film soundstage with black acoustic walls, an overhead lighting grid and a smooth concrete floor. It is a controlled modern production environment without natural light.",
    "location": {
      "lat": 52.216,
      "lon": 20.979
    },
    "permission_available": true,
    "power_available": true
  },
  {
    "id": 14,
    "name": "Attic Reading Room",
    "description": "A quiet attic library with low timber beams, shelves of books and warm desk lamps. Small dormer windows overlook tiled rooftops.",
    "location": {
      "lat": 52.242,
      "lon": 21.012
    },
    "permission_available": true,
    "power_available": false
  },
  {
    "id": 15,
    "name": "Garden Conservatory",
    "description": "A sunlit glass conservatory filled with tropical plants, tiled paths and wicker chairs. Curved windows frame a leafy garden.",
    "location": {
      "lat": 52.204,
      "lon": 21.036
    },
    "permission_available": true,
    "power_available": false
  },
  {
    "id": 16,
    "name": "Blue Apartment",
    "description": "A compact furnished apartment with blue kitchen cabinets, a modern sofa and a small balcony. White walls and practical furnishings create an everyday domestic setting.",
    "location": {
      "lat": 52.233,
      "lon": 21.02
    },
    "permission_available": true,
    "power_available": false
  },
  {
    "id": 17,
    "name": "Marble Lobby",
    "description": "A polished office lobby with marble columns, glass elevators and chrome reception desks. Its bright corporate interior has reflective surfaces and large modern windows.",
    "location": {
      "lat": 52.228,
      "lon": 21.006
    },
    "permission_available": true,
    "power_available": true
  },
  {
    "id": 18,
    "name": "Velvet Theatre",
    "description": "A small theatre with red velvet seats, a wooden stage and ornate ceiling moldings. Warm spotlights illuminate the empty auditorium.",
    "location": {
      "lat": 52.251,
      "lon": 21.004
    },
    "permission_available": true,
    "power_available": true
  },
  {
    "id": 19,
    "name": "Floodlit Court",
    "description": "An outdoor basketball court surrounded by wire fencing and apartment blocks. Painted lines and overhead floodlights define the urban sports setting.",
    "location": {
      "lat": 52.262,
      "lon": 21.017
    },
    "permission_available": true,
    "power_available": false
  },
  {
    "id": 20,
    "name": "Basement Club",
    "description": "A dark basement music club with low brick vaults, neon signs and a small dance floor. There are no windows and the ceiling is crowded with pipes.",
    "location": {
      "lat": 52.231,
      "lon": 20.999
    },
    "permission_available": true,
    "power_available": true
  },
  {
    "id": 21,
    "name": "Rooftop Terrace",
    "description": "An open rooftop terrace with concrete paving, railings and views across the city skyline. Modern office towers surround the exposed outdoor space.",
    "location": {
      "lat": 52.235,
      "lon": 20.99
    },
    "permission_available": true,
    "power_available": false
  },
  {
    "id": 22,
    "name": "Quiet Classroom",
    "description": "An empty classroom with wooden desks, a chalkboard and posters on pale walls. Broad windows light a clean educational interior.",
    "location": {
      "lat": 52.221,
      "lon": 21.025
    },
    "permission_available": true,
    "power_available": false
  },
  {
    "id": 23,
    "name": "Underground Garage",
    "description": "A concrete underground parking garage with low ceilings, fluorescent lights and painted support columns. Ramps connect rows of empty parking bays.",
    "location": {
      "lat": 52.228,
      "lon": 21.02
    },
    "permission_available": true,
    "power_available": true
  },
  {
    "id": 24,
    "name": "Greenhouse Yard",
    "description": "A fenced yard containing glass greenhouses, potting benches and stacks of terracotta pots. Natural light and climbing plants create a rural garden atmosphere.",
    "location": {
      "lat": 52.185,
      "lon": 21.087
    },
    "permission_available": true,
    "power_available": false
  },
  {
    "id": 25,
    "name": "Retro Laundrette",
    "description": "A narrow laundrette with tiled walls, rows of washing machines and plastic chairs. Bright signs and fluorescent lighting evoke an ordinary neighborhood shop.",
    "location": {
      "lat": 52.244,
      "lon": 21.039
    },
    "permission_available": true,
    "power_available": true
  },
  {
    "id": 26,
    "name": "Stone Courtyard",
    "description": "An enclosed historic courtyard with stone paving, ivy-covered walls and arched doorways. A small fountain sits between old residential facades.",
    "location": {
      "lat": 52.255,
      "lon": 21.01
    },
    "permission_available": true,
    "power_available": false
  },
  {
    "id": 27,
    "name": "Empty Hotel Corridor",
    "description": "A long carpeted hotel corridor with numbered doors, warm wall lamps and patterned wallpaper. The enclosed interior has an intimate, slightly dated appearance.",
    "location": {
      "lat": 52.226,
      "lon": 21.001
    },
    "permission_available": true,
    "power_available": false
  },
  {
    "id": 28,
    "name": "Workshop Annex",
    "description": "A working carpentry shop with brick walls, large windows, workbenches and neatly arranged hand tools. Sawdust covers the timber floor between machines.",
    "location": {
      "lat": 52.24,
      "lon": 20.961
    },
    "permission_available": true,
    "power_available": true
  },
  {
    "id": 29,
    "name": "Ceramic Gallery",
    "description": "A white-walled art gallery with polished wood flooring and carefully lit ceramic sculptures. Its renovated interior feels calm, bright and minimal.",
    "location": {
      "lat": 52.22,
      "lon": 21.016
    },
    "permission_available": true,
    "power_available": false
  },
  {
    "id": 30,
    "name": "Disused Office Wing",
    "description": "An abandoned office floor with peeling paint, damaged ceiling tiles and rows of small windows. Empty desks and worn carpet create a neglected institutional interior.",
    "location": {
      "lat": 52.263,
      "lon": 20.981
    },
    "permission_available": true,
    "power_available": false
  },
  {
    "id": 31,
    "name": "Freight Shed",
    "description": "An old freight shed with corrugated metal cladding, steel beams and sliding doors. The interior is empty and dusty, with little daylight and no brick walls.",
    "location": {
      "lat": 52.288,
      "lon": 21.101
    },
    "permission_available": true,
    "power_available": true
  },
  {
    "id": 32,
    "name": "Boiler Room",
    "description": "A decommissioned industrial boiler room with pipes, metal walkways and soot-darkened brick walls. Narrow high windows illuminate rusting machinery.",
    "location": {
      "lat": 52.269,
      "lon": 21.078
    },
    "permission_available": true,
    "power_available": false
  },
  {
    "id": 33,
    "name": "Daylight Furniture Loft",
    "description": "A renovated furniture showroom in a former warehouse with exposed brick, tall factory windows and a pale wooden floor. Designer sofas and tables fill the bright, carefully maintained interior.",
    "location": {
      "lat": 52.236,
      "lon": 20.976
    },
    "permission_available": true,
    "power_available": true
  },
  {
    "id": 34,
    "name": "Covered Market",
    "description": "A busy indoor market hall with brick arches, glazed roof panels and colorful stalls. Signs, produce and people fill the lively public interior.",
    "location": {
      "lat": 52.258,
      "lon": 21.027
    },
    "permission_available": false,
    "power_available": true
  },
  {
    "id": 35,
    "name": "Empty Pool",
    "description": "A drained indoor swimming pool with blue tiles, changing-room doors and high clerestory windows. Echoes and faded lane markings suggest a closed sports facility.",
    "location": {
      "lat": 52.2,
      "lon": 20.976
    },
    "permission_available": true,
    "power_available": true
  },
  {
    "id": 36,
    "name": "Gravel Service Yard",
    "description": "An outdoor industrial yard with gravel, shipping containers and chain-link fencing. Steel loading equipment stands beside a low utility building.",
    "location": {
      "lat": 52.287,
      "lon": 20.939
    },
    "permission_available": true,
    "power_available": false
  },
  {
    "id": 37,
    "name": "Timber Barn",
    "description": "An old wooden barn with hay bales, exposed rafters and sunlight through gaps in the boards. Its rough rural interior has an earth floor and no industrial equipment.",
    "location": {
      "lat": 52.145,
      "lon": 20.923
    },
    "permission_available": true,
    "power_available": false
  },
  {
    "id": 38,
    "name": "Control Room",
    "description": "A retro control room with analog gauges, switch panels and worn operator chairs. Narrow windows overlook a factory floor, but the room itself is compact and enclosed.",
    "location": {
      "lat": 52.274,
      "lon": 21.055
    },
    "permission_available": true,
    "power_available": true
  },
  {
    "id": 39,
    "name": "White Dance Hall",
    "description": "A clean dance rehearsal hall with mirrors, white walls and a sprung wooden floor. Large windows provide even daylight across the open modern room.",
    "location": {
      "lat": 52.219,
      "lon": 21.009
    },
    "permission_available": true,
    "power_available": true
  },
  {
    "id": 40,
    "name": "Unmapped Factory",
    "description": "An abandoned industrial warehouse with exposed brick walls, tall factory windows and an empty concrete floor. Its location has not yet been verified in the catalog.",
    "permission_available": true,
    "power_available": true
  }
]""")
assert len(locations) == 40
print(f"{len(locations)} fictional film locations loaded")

40 fictional film locations loaded


## 4. Embed Descriptions and Create the Collection

FastEmbed runs the small English BGE model on the CPU. We pin its ONNX repository revision as well as the Python package. `passage_embed` embeds catalog text and `query_embed` embeds the scout's request. Coordinates and facility flags are not included in the embedding text; their later contribution stays visible.

The collection uses cosine similarity. Create payload indexes for the two Boolean fields and the geographic field before loading points. The notebook uses a fresh collection in its isolated server, so rerunning this cell safely replaces only its own demo data.

In [4]:
from fastembed import TextEmbedding
from huggingface_hub import snapshot_download

MODEL_NAME = "BAAI/bge-small-en-v1.5"
MODEL_REVISION = "aa8f8b060edb00e03bfdd08813a2949946c8ba55"
model_path = snapshot_download(
    repo_id="qdrant/bge-small-en-v1.5-onnx-q",
    revision=MODEL_REVISION,
    allow_patterns=["*.json", "*.txt", "model_optimized.onnx"],
    cache_dir=str(runtime_dir / "model-cache"),
)
encoder = TextEmbedding(
    model_name=MODEL_NAME,
    specific_model_path=model_path,
    threads=2,
    providers=["CPUExecutionProvider"],
)
embeddings = list(encoder.passage_embed(item["description"] for item in locations))
QUERY_TEXT = (
    "An abandoned industrial warehouse with exposed brick walls and tall windows"
)
query_vector = next(iter(encoder.query_embed(QUERY_TEXT))).tolist()

COLLECTION = "film_locations"
if await client.collection_exists(COLLECTION):
    await client.delete_collection(COLLECTION)
await client.create_collection(
    collection_name=COLLECTION,
    vectors_config=models.VectorParams(
        size=len(query_vector), distance=models.Distance.COSINE
    ),
)
for field, schema in [
    ("permission_available", models.PayloadSchemaType.BOOL),
    ("power_available", models.PayloadSchemaType.BOOL),
    ("location", models.PayloadSchemaType.GEO),
]:
    await client.create_payload_index(COLLECTION, field, field_schema=schema, wait=True)
await client.upsert(
    collection_name=COLLECTION,
    points=[
        models.PointStruct(
            id=item["id"],
            vector=embedding.tolist(),
            payload={key: value for key, value in item.items() if key != "id"},
        )
        for item, embedding in zip(locations, embeddings, strict=True)
    ],
    wait=True,
)
print(
    f"Indexed {(await client.count(COLLECTION, exact=True)).count} points, {len(query_vector)} dimensions"
)

Indexed 40 points, 384 dimensions


## 5. Inspect Semantic Retrieval Before Changing the Score

First, retrieve by description alone. The output makes permission, location, and power visible so you can distinguish a good visual match from a usable location. Cosine scores are similarity values, not probabilities.

For this small catalog we use exact vector search and retrieve all 40 candidates. That makes the ranking demonstration reproducible without an approximate-nearest-neighbor effect. A larger collection needs a measured candidate budget and appropriate ANN settings.

In [5]:
import math

import pandas as pd
from IPython.display import display

BASE_LOCATION = models.GeoPoint(lat=52.2297, lon=21.0122)
CANDIDATE_LIMIT = len(locations)
TOP_K = 5
EXACT = models.SearchParams(exact=True)


def distance_m(origin: models.GeoPoint, destination: dict[str, float]) -> float:
    lat1, lat2 = math.radians(origin.lat), math.radians(destination["lat"])
    dlat = lat2 - lat1
    dlon = math.radians(destination["lon"] - origin.lon)
    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    )
    return 6_371_000 * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def name_of(point: models.ScoredPoint) -> str:
    assert point.payload is not None
    return str(point.payload["name"])


def show_hits(
    points: list[models.ScoredPoint], origin: models.GeoPoint = BASE_LOCATION
) -> pd.DataFrame:
    rows = []
    for point in points:
        payload = point.payload or {}
        location = payload.get("location")
        rows.append(
            {
                "id": point.id,
                "name": payload["name"],
                "score": point.score,
                "distance_km": distance_m(origin, location) / 1000
                if location
                else None,
                "permission": payload["permission_available"],
                "power": payload["power_available"],
            }
        )
    return pd.DataFrame(rows).round(4)


baseline = (
    await client.query_points(
        collection_name=COLLECTION,
        query=query_vector,
        search_params=EXACT,
        limit=CANDIDATE_LIMIT,
        with_payload=True,
    )
).points
semantic_scores = {point.id: point.score for point in baseline}
display(show_hits(baseline[:TOP_K]))

,id,name,score,distance_km,permission,power
0,11,Outer Ring Hangar,0.9060,19.9376,True,True
1,40,Unmapped Factory,0.8968,NaN,True,True
2,4,North Foundry,0.8773,9.3882,True,True
3,1,Old Kiln Hall,0.8686,3.4479,False,True
4,2,East Brickworks,0.8686,6.5158,True,False


## 6. Apply the Hard Requirements

The crew can consider only locations with permission available and within **15 km of the base**. These rules belong in a filter. A larger score must never make an unavailable venue eligible.

`geo_radius` is a geographic radius, not a driving-time estimate. The same applies to the distance used by the scoring formula later.

In [6]:
MAX_DISTANCE_M = 15_000.0


def make_eligibility_filter(origin: models.GeoPoint) -> models.Filter:
    return models.Filter(
        must=[
            models.FieldCondition(
                key="permission_available", match=models.MatchValue(value=True)
            ),
            models.FieldCondition(
                key="location",
                geo_radius=models.GeoRadius(center=origin, radius=MAX_DISTANCE_M),
            ),
        ]
    )


eligibility_filter = make_eligibility_filter(BASE_LOCATION)
filtered = (
    await client.query_points(
        collection_name=COLLECTION,
        query=query_vector,
        query_filter=eligibility_filter,
        search_params=EXACT,
        limit=CANDIDATE_LIMIT,
        with_payload=True,
    )
).points
display(show_hits(filtered[:TOP_K]))
print(f"Eligible candidates: {len(filtered)} / {len(locations)}")
assert {1, 11, 40}.isdisjoint(point.id for point in filtered)

Eligible candidates: 36 / 40


,id,name,score,distance_km,permission,power
0,4,North Foundry,0.8773,9.3882,True,True
1,2,East Brickworks,0.8686,6.5158,True,False
2,6,Trackside Depot,0.8250,8.3122,True,False
3,7,South Textile Mill,0.7819,11.6263,True,True
4,9,West Loading Hall,0.7620,7.0352,True,True


## 7. Prefer a Useful Facility

A venue with on-site power saves setup work, but power is a preference for this shoot. A matching Boolean condition evaluates to 1; a non-match evaluates to 0. Add `0.08 × power_available` to each candidate's semantic score.

The formula rescores only the candidates returned by `prefetch`. The eligibility filter sits inside that prefetch, so its candidate budget is spent on eligible locations.

In [7]:
FACILITY_WEIGHT = 0.08
power_condition = models.FieldCondition(
    key="power_available", match=models.MatchValue(value=True)
)
facility_bonus = models.MultExpression(mult=[FACILITY_WEIGHT, power_condition])

facility_boost = (
    await client.query_points(
        collection_name=COLLECTION,
        prefetch=models.Prefetch(
            query=query_vector,
            filter=eligibility_filter,
            params=EXACT,
            limit=CANDIDATE_LIMIT,
        ),
        query=models.FormulaQuery(
            formula=models.SumExpression(sum=["$score", facility_bonus])
        ),
        limit=TOP_K,
        with_payload=True,
    )
).points
display(show_hits(facility_boost))

,id,name,score,distance_km,permission,power
0,4,North Foundry,0.9573,9.3882,True,True
1,2,East Brickworks,0.8686,6.5158,True,False
2,7,South Textile Mill,0.8619,11.6263,True,True
3,9,West Loading Hall,0.8420,7.0352,True,True
4,3,Riverside Workshop,0.8345,1.8063,True,True


## 8. Prefer Closer Matches With a Gradual Decay

Distance is a softer preference than the 15 km boundary. We use Gaussian decay: proximity is 1 at the crew base, 0.5 at 5 km, and 0.0625 at 10 km. `scale=5000` expresses the distance where the bonus halves; it does not impose a 5 km cutoff.

Our bonus is `0.15 × proximity`. It adds at most 0.15 score units. These weights are teaching choices; evaluate them against labeled search requests before using them in a product.

In [8]:
GEO_WEIGHT = 0.15
DISTANCE_SCALE_M = 5_000.0


def make_proximity(origin: models.GeoPoint) -> models.GaussDecayExpression:
    return models.GaussDecayExpression(
        gauss_decay=models.DecayParamsExpression(
            x=models.GeoDistance(
                geo_distance=models.GeoDistanceParams(origin=origin, to="location")
            ),
            target=0.0,
            scale=DISTANCE_SCALE_M,
            midpoint=0.5,
        )
    )


proximity = make_proximity(BASE_LOCATION)
proximity_boost = (
    await client.query_points(
        collection_name=COLLECTION,
        prefetch=models.Prefetch(
            query=query_vector,
            filter=eligibility_filter,
            params=EXACT,
            limit=CANDIDATE_LIMIT,
        ),
        query=models.FormulaQuery(
            formula=models.SumExpression(
                sum=[
                    "$score",
                    models.MultExpression(mult=[GEO_WEIGHT, proximity]),
                ]
            )
        ),
        limit=TOP_K,
        with_payload=True,
    )
).points
display(show_hits(proximity_boost))

,id,name,score,distance_km,permission,power
0,2,East Brickworks,0.9148,6.5158,True,False
1,3,Riverside Workshop,0.8915,1.8063,True,True
2,4,North Foundry,0.8903,9.3882,True,True
3,6,Trackside Depot,0.8471,8.3122,True,False
4,33,Daylight Furniture Loft,0.8459,2.5629,True,True


## 9. Combine the Signals and Explain the Result

The final formula is:

`score = cosine_similarity + 0.08 × has_power + 0.15 × proximity`

The helper below keeps the same filtered retrieval stage while making weights, origin, and candidate limit easy to experiment with. The explanation table reconstructs the two bonuses from the stored payload and checks the result against the server's actual score. Small floating-point differences are expected.

In [9]:
async def rank_locations(
    origin: models.GeoPoint = BASE_LOCATION,
    facility_weight: float = FACILITY_WEIGHT,
    geo_weight: float = GEO_WEIGHT,
    candidate_limit: int = CANDIDATE_LIMIT,
    limit: int = TOP_K,
) -> list[models.ScoredPoint]:
    response = await client.query_points(
        collection_name=COLLECTION,
        prefetch=models.Prefetch(
            query=query_vector,
            filter=make_eligibility_filter(origin),
            params=EXACT,
            limit=candidate_limit,
        ),
        query=models.FormulaQuery(
            formula=models.SumExpression(
                sum=[
                    "$score",
                    models.MultExpression(mult=[facility_weight, power_condition]),
                    models.MultExpression(mult=[geo_weight, make_proximity(origin)]),
                ]
            )
        ),
        limit=limit,
        with_payload=True,
    )
    return response.points


combined = await rank_locations()
display(show_hits(combined))

,id,name,score,distance_km,permission,power
0,3,Riverside Workshop,0.9715,1.8063,True,True
1,4,North Foundry,0.9703,9.3882,True,True
2,33,Daylight Furniture Loft,0.9259,2.5629,True,True
3,2,East Brickworks,0.9148,6.5158,True,False
4,8,Rehearsal Loft,0.8807,2.6704,True,True


In [10]:
def explain_scores(
    points: list[models.ScoredPoint],
    origin: models.GeoPoint = BASE_LOCATION,
    facility_weight: float = FACILITY_WEIGHT,
    geo_weight: float = GEO_WEIGHT,
) -> pd.DataFrame:
    rows = []
    for point in points:
        payload = point.payload or {}
        distance = distance_m(origin, payload["location"])
        geo_bonus = geo_weight * math.exp(
            math.log(0.5) * (distance / DISTANCE_SCALE_M) ** 2
        )
        power_bonus = facility_weight * float(payload["power_available"])
        reconstructed = semantic_scores[point.id] + power_bonus + geo_bonus
        assert math.isclose(reconstructed, point.score, abs_tol=2e-5)
        rows.append(
            {
                "name": payload["name"],
                "semantic": semantic_scores[point.id],
                "distance_km": distance / 1000,
                "power_bonus": power_bonus,
                "geo_bonus": geo_bonus,
                "final_score": point.score,
            }
        )
    return pd.DataFrame(rows).round(4)


display(explain_scores(combined))
comparison = pd.DataFrame(
    {
        "rank": range(1, TOP_K + 1),
        "semantic + eligibility": [name_of(p) for p in filtered[:TOP_K]],
        "power bonus": [name_of(p) for p in facility_boost],
        "geo bonus": [name_of(p) for p in proximity_boost],
        "combined": [name_of(p) for p in combined],
    }
)
display(comparison)

,name,semantic,distance_km,power_bonus,geo_bonus,final_score
0,Riverside Workshop,0.7545,1.8063,0.08,0.1370,0.9715
1,North Foundry,0.8773,9.3882,0.08,0.0130,0.9703
2,Daylight Furniture Loft,0.7209,2.5629,0.08,0.1250,0.9259
3,East Brickworks,0.8686,6.5158,0.00,0.0462,0.9148
4,Rehearsal Loft,0.6776,2.6704,0.08,0.1231,0.8807


,rank,semantic + eligibility,power bonus,geo bonus,combined
0,1,North Foundry,North Foundry,East Brickworks,Riverside Workshop
1,2,East Brickworks,East Brickworks,Riverside Workshop,North Foundry
2,3,Trackside Depot,South Textile Mill,North Foundry,Daylight Furniture Loft
3,4,South Textile Mill,West Loading Hall,Trackside Depot,East Brickworks
4,5,West Loading Hall,Riverside Workshop,Daylight Furniture Loft,Rehearsal Loft


## 10. Check the Boundaries Before Tuning

Turning both bonuses off should recover the eligible semantic ranking. All ranked results must still have permission and lie within the radius. The deliberately unmapped point must stay excluded. A radius with no matching locations should return an empty list, not fabricate a recommendation.

These checks verify the mechanics on this fixture. They do not establish that the weights improve user satisfaction or production search quality.

In [11]:
without_bonuses = await rank_locations(
    facility_weight=0.0, geo_weight=0.0, limit=CANDIDATE_LIMIT
)
assert [p.id for p in without_bonuses] == [p.id for p in filtered]
for actual, expected in zip(without_bonuses, filtered, strict=True):
    assert math.isclose(actual.score, expected.score, abs_tol=1e-6)
for point in await rank_locations(limit=CANDIDATE_LIMIT):
    payload = point.payload or {}
    assert payload["permission_available"] is True
    assert distance_m(BASE_LOCATION, payload["location"]) <= MAX_DISTANCE_M
    assert point.id not in {1, 11, 40}
empty_filter = models.Filter(
    must=[
        models.FieldCondition(
            key="location",
            geo_radius=models.GeoRadius(center=BASE_LOCATION, radius=1.0),
        )
    ]
)
empty = (
    await client.query_points(
        collection_name=COLLECTION,
        prefetch=models.Prefetch(
            query=query_vector, filter=empty_filter, limit=CANDIDATE_LIMIT
        ),
        query=models.FormulaQuery(
            formula=models.SumExpression(sum=["$score", facility_bonus])
        ),
        limit=TOP_K,
    )
).points
assert empty == []
print(
    "Passed: zero bonuses, eligibility, missing coordinates, score arithmetic, and empty result"
)

Passed: zero bonuses, eligibility, missing coordinates, score arithmetic, and empty result


## 11. Two Ways to Make the Ranking Worse

**Overweight proximity.** Raising the distance coefficient to 5 makes small distance differences overwhelm semantic differences. Inspect the winning descriptions rather than treating a larger final number as a better result. A combined score is an ordering rule, not an accuracy measure.

In [12]:
overweighted = await rank_locations(geo_weight=5.0)
display(explain_scores(overweighted, geo_weight=5.0))
assert overweighted[0].payload is not None
print("Winner's description:", overweighted[0].payload["description"])

Winner's description: A bright contemporary photography studio with white plaster walls, a seamless backdrop and a polished floor. The room has modern lighting and a clean commercial appearance.


,name,semantic,distance_km,power_bonus,geo_bonus,final_score
0,Glass Studio,0.5786,0.0639,0.08,4.9994,5.6580
1,Underground Garage,0.6210,0.5639,0.08,4.9561,5.6571
2,Marble Lobby,0.5592,0.4626,0.08,4.9704,5.6096
3,Basement Club,0.5813,0.9105,0.08,4.8864,5.5477
4,Blue Apartment,0.5598,0.6456,0.00,4.9425,5.5024


**Retrieve too few candidates.** Fetching only the three best semantic matches can omit a location that the final formula would have promoted. The formula can only reorder what prefetch returns; increasing a bonus cannot bring an omitted point back.

For this demonstration, compare the same formula with prefetch limits of 3 and 40. The latter covers all 36 eligible records in this fixture. In a larger system, tune candidate count using relevance labels and latency measurements together.

In [13]:
narrow = await rank_locations(candidate_limit=3, limit=3)
wide = await rank_locations(candidate_limit=CANDIDATE_LIMIT, limit=3)
display(
    pd.DataFrame(
        {
            "rank": [1, 2, 3],
            "prefetch limit 3": [name_of(p) for p in narrow],
            "prefetch limit 40": [name_of(p) for p in wide],
        }
    )
)
omitted = [
    name_of(p) for p in wide if p.id not in {candidate.id for candidate in narrow}
]
print("Promoted locations the narrow retrieval missed:", omitted)
if not omitted:
    print("Both candidate budgets returned the same top results for this request.")

Promoted locations the narrow retrieval missed: ['Riverside Workshop', 'Daylight Furniture Loft']


,rank,prefetch limit 3,prefetch limit 40
0,1,North Foundry,Riverside Workshop
1,2,East Brickworks,North Foundry
2,3,Trackside Depot,Daylight Furniture Loft


## 12. Move the Crew Base

The semantic request is unchanged. Move the base north, then inspect how distance bonuses and the 15 km eligibility boundary change. The same query vector can be reused; we only change geographic inputs.

**Try it:** change `NEW_BASE` and rerun the next cell. Before running a change, predict which location should rise and which should fall. Can you explain the result from the components in the table?

In [14]:
NEW_BASE = models.GeoPoint(lat=52.310, lon=21.040)
moved = await rank_locations(origin=NEW_BASE)
if not moved:
    print("No eligible locations near this base. Try another origin or radius.")
else:
    display(explain_scores(moved, origin=NEW_BASE))
    print("Original winner:", name_of(combined[0]))
    print("New-base winner:", name_of(moved[0]))
    if [p.id for p in moved] == [p.id for p in combined]:
        print("The ranking is unchanged; inspect the distance contributions.")


Original winner: Riverside Workshop
New-base winner: North Foundry


,name,semantic,distance_km,power_bonus,geo_bonus,final_score
0,North Foundry,0.8773,0.3017,0.08,0.1496,1.1069
1,Outer Ring Hangar,0.9060,11.1490,0.08,0.0048,0.9907
2,East Brickworks,0.8686,6.9242,0.00,0.0397,0.9083
3,Riverside Workshop,0.7545,8.0133,0.08,0.0253,0.8598
4,Canal Storehouse,0.7442,3.4254,0.00,0.1083,0.8526


## 13. Tune the Weights Explicitly

Pass trial weights to both ranking and explanation, so the displayed components describe the exact formula you executed. Python captures a function's default arguments when it is defined: changing a global weight alone does not update those defaults.

You can also change `DISTANCE_SCALE_M`, which the proximity helper reads when called, then rerun this cell. If you change the scene description, regenerate `query_vector` with `encoder.query_embed`, rerun the baseline cell to rebuild `semantic_scores`, and rerun the filtered and scoring cells. This keeps every comparison about the same request.

In [15]:
trial_facility_weight = 0.05
trial_geo_weight = 0.10
trial = await rank_locations(
    facility_weight=trial_facility_weight,
    geo_weight=trial_geo_weight,
)
display(
    explain_scores(
        trial,
        facility_weight=trial_facility_weight,
        geo_weight=trial_geo_weight,
    )
)

,name,semantic,distance_km,power_bonus,geo_bonus,final_score
0,North Foundry,0.8773,9.3882,0.05,0.0087,0.9360
1,East Brickworks,0.8686,6.5158,0.00,0.0308,0.8994
2,Riverside Workshop,0.7545,1.8063,0.05,0.0914,0.8958
3,Daylight Furniture Loft,0.7209,2.5629,0.05,0.0834,0.8542
4,Trackside Depot,0.8250,8.3122,0.00,0.0147,0.8397


## What to Change for a Real Catalog

- Replace the fictional records with permissioned, maintained catalog data. Treat coordinates, filming permission, and facilities as independently verified fields.
- Keep non-negotiable restrictions in filters. Use score bonuses only for preferences.
- Retain a sufficiently broad candidate pool. Measure relevant-location recall before tuning final ordering.
- Evaluate several queries and crew bases with human relevance judgments. Track candidate recall, ranking quality, and latency; do not infer improvement from this small example.
- Geographic distance does not account for roads, bridges, travel time, or vehicle access. Use a separately maintained travel-time signal when that is the actual requirement.
- Similarity distributions depend on the embedding model and distance metric. Recalibrate weights if either changes, or if you introduce hybrid retrieval.

References: [Score boosting and decay](https://qdrant.tech/documentation/search/search-relevance/), [multi-stage queries](https://qdrant.tech/documentation/search/hybrid-queries/), [geographic filters](https://qdrant.tech/documentation/search/filtering/), [payload indexes](https://qdrant.tech/documentation/manage-data/indexing/), [server release](https://github.com/qdrant/qdrant/releases/tag/v1.20.2), [pinned model](https://huggingface.co/qdrant/bge-small-en-v1.5-onnx-q/tree/aa8f8b060edb00e03bfdd08813a2949946c8ba55).

## Optional Cleanup

Set `CLEAN_UP=True` when you have finished experimenting. This stops this notebook's subprocess and removes only its temporary directory. The process is also registered to terminate when the Python kernel exits.

In [16]:
CLEAN_UP = False
if CLEAN_UP:
    await client.close()
    server.terminate()
    server.wait(timeout=10)
    server_log.close()
    shutil.rmtree(runtime_dir)
    print("Stopped the tutorial server and removed its temporary data")